## Defining a job — UI, REST API & YAML

Three ways to define a job. The exam doesn't ask for exact syntax, but it does expect you to recognise the **YAML shape**, because that's how Automation Bundles (module 07) deploy jobs.

- **UI** — point-and-click; fine for one-offs and exploration.
- **Jobs REST API** — `POST /api/2.1/jobs/create` with a JSON body; the same JSON that Automation Bundles emit.
- **YAML in an Automation Bundle** — the **production pattern**; version-controlled, promoted dev → prod.

A tiny bundle-job shape (full structure lives in module 07):

```yaml
resources:
  jobs:
    fintech_nightly_ingest:
      schedule:
        quartz_cron_expression: "0 0 2 * * ?"
        timezone_id: "UTC"
      email_notifications:
        on_failure: ["oncall@bank.example"]
      tasks:
        - task_key: ingest_cards
          notebook_task: { notebook_path: ../notebooks/ingest_cards.ipynb }
          job_cluster_key: shared_etl_cluster
          max_retries: 3
        - task_key: notify
          depends_on: [{ task_key: ingest_cards }]
          run_if: ALL_DONE
          notebook_task: { notebook_path: ../notebooks/notify.ipynb }
```

**Read the structure, not the syntax.** Everything from this module is visible in that YAML: the DAG (`depends_on`), the trigger (`schedule`), the retries (`max_retries`), and the always-fires notification (`run_if: ALL_DONE`). It's the same job the UI shows — just declared as code, which is exactly where module 07 (CI/CD & Automation Bundles) picks up.
